In [1]:
# Pre-training SoRL on arithmatic generalization dataset
# ------------------------------------------------------ 
import torch
from sorl.gat_sim import GAT, GATConfig
torch.set_float32_matmul_precision('high')  # Enable TF32 for ~2x speedup

BOS_TOKEN_ID = 20
gat_config = GATConfig(
    vocab_sizes=[BOS_TOKEN_ID+1, 16],  # 16 abstract tokens
    n_layer=4,
    n_head=4,
    n_embd=128,
    device="cuda" if torch.cuda.is_available() else "cpu",
    # bos_token_id=BOS_TOKEN_ID
)
    
model = GAT(gat_config)
# model = model.to("cuda")
# model = torch.compile(model)

Generate multiplication data: 

```python data/arithmetic.py```

In [2]:
# ---- Arithmatic generalization dataset loader ----
from sorl.arithmetic import data_generator
from sorl.arithmetic import DigitTokenizer, ArithmeticDataLoader

# --- tokenizer ---
tokenizer = DigitTokenizer(BOS_TOKEN_ID)
# --- Multiplication Datasset loader
loader = ArithmeticDataLoader(min_digits=1, max_digits=3, num_examples=800, pad_digits=6, device=model.device)


100%|██████████| 800/800 [00:00<00:00, 48406.52it/s]


In [ ]:
# Information Gain Fromulation
# ------------------------------------------------------------------------
from sorl.neo_utils import sorl_evaluate, sorl_search_v8
from sorl.topo import orthogonalize_abs_param
from collections import defaultdict
from sorl.info import SoRLLoss

# --- orthogonal initialization on abs param --- 
orthogonalize_abs_param(model, do_wte=True, do_head=True)

K = 4
max_iterations = 2
loss_fn = SoRLLoss(model.vocab_sizes[1], decay=0.8, target_vocab_util=0.9)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
n = 2
temperature = torch.tensor([0.0, 5.0], device=model.device)
num_steps = 400
alpha_abs = 0.1
alpha_soft_zipf = 1.0
alpha_info_gain = 10.0
attn_blocksize = 1792
phase = "compression"
memory_span = 1792 

record = defaultdict(list)
img_frames = []

for step in range(num_steps): 

    optimizer.zero_grad()

    tokens, doc_ids = loader.get_batch(4)

    with torch.no_grad(): 
        best_data, best_traj_ppt, best_abs_ppt, search_adv = sorl_search_v8(tokens, model, n=n, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperature, truncate_seq_len=False)

    # --- compute loss --- 
    base_traj_loss, base_logits = model.forward(tokens, memory_span, attn_blocksize)
    base_traj_loss = base_traj_loss.mean()
    info_gain_loss, abs_loss, zipf_bigram_loss = loss_fn(best_data, model, base_traj_loss.detach(), memory_span, attn_blocksize)
    loss = base_traj_loss + alpha_info_gain * info_gain_loss.mean() + alpha_abs * abs_loss + alpha_soft_zipf * zipf_bigram_loss

    # --- log relative info gain ---
    rel_info_gain = ((-info_gain_loss) / base_traj_loss).detach()

    # --- optimize --- 
    loss.backward() 
    optimizer.step()

    if step % 2 == 0: 
        with torch.no_grad(): 
            temperatures_eval = torch.tensor([0.0, 10.0], device=model.device)

            val_tokens, val_adv, traj_loss, abs_loss = sorl_evaluate(tokens, model, n=2, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=attn_blocksize, temperature=temperatures_eval,
                                                                     truncate_seq_len=False)
            # _, _, zipf_bigram_loss = loss_fn(val_tokens, model, memory_span_abs, memory_span_traj, attn_blocksize)
     

        # print(f"\n{phase} | step {step} | base traj loss: {base_traj_loss.item():.2f} | cond traj loss: {traj_loss.mean().item():.2f} | rel search info gain: {rel_info_gain * 100:.2f}% | greedy adv: {val_adv.item() * 100:.2f}% | vocab util: {abs_stats.vocab_util * 100:.2f}%  | avg logit sim: {avg_logit_sim:.2f} |  bigram-zipf kl: {zipf_bigram_loss.item():.2f} | bigram rep rate: {abs_stats.bigram_rep_rate:.2f} | rel info gain (search): {rel_info_gain:.2f}")
        print("Step", step)
        # img = visualize_dynamics(abs_stats, loader, model, enc, K, step)
        # img_frames.append(img)
        # break

Step 0
Step 2
Step 4
Step 6
Step 8
Step 10
Step 12
Step 14


In [11]:
# generate function implementation 
# ----------------------------------
from sorl.neo_utils import generate
from sorl.arithmetic import process_query, check_answer

K = 5
tokens = next(val_loader)
idx, answer_idx = process_query(tokens)

print(f"init   | idx: {idx[0].tolist()} | question: {tokenizer.decode(idx[0].tolist()[1:-1])}")
for i in range(len(answer_idx[0])*2): 
    idx = generate(model, idx, K=K, max_iterations=max_iterations, memory_span=memory_span, attn_blocksize=1792, 
                   temperature=temperatures_eval[0])
    idx_without_abstraction = idx[idx < model.vocab_sizes[0]]
    print(f"step {i+1} idx (abstraction free): {idx_without_abstraction.tolist()}")
    print(f"                         idx : {idx[0].tolist()}")

is_correct, pred_answer, true_answer = check_answer(idx_without_abstraction, answer_idx, tokenizer)
print(f"is_correct: {is_correct} | pred_answer: {pred_answer} | true_answer: {true_answer}")

init   | idx: [0, 15, 4, 2, 4, 18, 4, 3] | question: 5 x 8 
step 1 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4]
                         idx : [0, 15, 4, 2, 4, 27, 18, 4, 3, 4]
step 2 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 11]
                         idx : [0, 15, 4, 2, 4, 27, 18, 4, 3, 4, 11]
step 3 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 11, 16]
                         idx : [0, 15, 4, 2, 4, 27, 18, 4, 3, 4, 11, 16]
step 4 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 11, 16, 1]
                         idx : [0, 15, 4, 2, 4, 27, 18, 4, 3, 4, 11, 25, 16, 1]
step 5 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 11, 16, 1, 0]
                         idx : [0, 15, 4, 2, 4, 27, 18, 4, 3, 4, 11, 25, 16, 1, 0]
step 6 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 11, 16, 1, 0, 17]
                         idx : [0, 15, 4, 2, 4, 27, 18, 4, 3, 4, 11, 25, 16, 1, 0, 17]
step 7 idx (abstraction free): [0, 15, 4, 2, 4, 18, 4, 3, 4, 1